# Oracle on Azure Workshop - Admin Notebook

Dieses Notebook ermöglicht:
- SSH-Verbindung zur User00 VM via Service Principal
- Verbindung zur Oracle Autonomous Database (ADB) im ODAA VNet
- Netzwerk-Diagnose und Tests

## Helper Functions & Configuration

In [1]:
# Helper functions for Oracle on Azure Workshop
import subprocess
import sys
import json
import os
import re

# Determine directories relative to notebook location
NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
TERRAFORM_DIR = os.path.dirname(NOTEBOOK_DIR) if os.path.basename(NOTEBOOK_DIR) == "notebook" else NOTEBOOK_DIR
if not os.path.exists(os.path.join(TERRAFORM_DIR, "terraform.tfvars")):
    # Fallback: look in parent
    TERRAFORM_DIR = os.path.dirname(TERRAFORM_DIR)
    
print(f"Terraform dir: {TERRAFORM_DIR}")

def run(cmd, shell=True, cwd=None, timeout=None):
    """Run command and print output. Returns exit code."""
    result = subprocess.run(
        cmd,
        shell=shell,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        cwd=cwd or TERRAFORM_DIR,
        timeout=timeout,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    return result.returncode

def run_output(cmd, shell=True, cwd=None, timeout=None):
    """Run command and return stdout as string."""
    result = subprocess.run(
        cmd,
        shell=shell,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        cwd=cwd or TERRAFORM_DIR,
        timeout=timeout,
    )
    return result.stdout.strip() if result.stdout else ""

def run_args(args, cwd=None, timeout=None):
    """Run a command without shell."""
    result = subprocess.run(
        args,
        shell=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        cwd=cwd or TERRAFORM_DIR,
        timeout=timeout,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    return result.returncode

def az_json(cmd):
    """Run az CLI command and return parsed JSON."""
    raw = run_output(cmd)
    try:
        return json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return None

def parse_tfvars(path):
    """Parse terraform.tfvars file."""
    values = {}
    if not os.path.exists(path):
        return values
    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or line.startswith("//"):
                continue
            line = re.split(r"\s+#", line, maxsplit=1)[0].strip()
            m = re.match(r"^(?P<key>[A-Za-z0-9_]+)\s*=\s*(?P<val>.+)$", line)
            if not m:
                continue
            key = m.group("key")
            val = m.group("val").strip()
            if val.startswith('"') and val.endswith('"'):
                values[key] = val[1:-1]
    return values

print("✅ Helper functions loaded.")
print(f"   Terraform dir: {TERRAFORM_DIR}")

Terraform dir: c:\Users\chpinoto\workspace\msftmh\03-Azure\01-03-Infrastructure\10_Oracle_on_Azure\resources\infra\terraform
✅ Helper functions loaded.
   Terraform dir: c:\Users\chpinoto\workspace\msftmh\03-Azure\01-03-Infrastructure\10_Oracle_on_Azure\resources\infra\terraform


## Service Principal Login

Authentifizierung mit dem Service Principal aus `terraform.tfvars`.
Der SP hat bereits `Virtual Machine Administrator Login` Rechte auf Subscription-Ebene.

In [ ]:
# ============================================================================
# Service Principal Authentication (from terraform.tfvars)
# ============================================================================

TFVARS_PATH = os.path.join(TERRAFORM_DIR, "terraform.tfvars")
NOTEBOOK_AZURE_CONFIG = os.path.join(TERRAFORM_DIR, ".azure-notebook")
os.makedirs(NOTEBOOK_AZURE_CONFIG, exist_ok=True)
os.environ["AZURE_CONFIG_DIR"] = NOTEBOOK_AZURE_CONFIG

tfvars = parse_tfvars(TFVARS_PATH)

# Check required variables (matching terraform.tfvars variable names)
required = ["client_id", "client_secret", "tenant_id", "vm_subscription_id"]
missing = [k for k in required if not tfvars.get(k)]
if missing:
    raise ValueError(f"Fehlende Felder in terraform.tfvars: {', '.join(missing)}")

CLIENT_ID = tfvars["client_id"]
CLIENT_SECRET = tfvars["client_secret"]
TENANT_ID = tfvars["tenant_id"]
VM_SUBSCRIPTION_ID = tfvars["vm_subscription_id"]
ODAA_SUBSCRIPTION_ID = tfvars.get("odaa_subscription_id", "")
LOCATION = tfvars.get("location", "francecentral")

print(f"✅ Konfiguration geladen")
print(f"   Tenant ID: {TENANT_ID}")
print(f"   VM Subscription: {VM_SUBSCRIPTION_ID}")
print(f"   ODAA Subscription: {ODAA_SUBSCRIPTION_ID}")
print(f"   Location: {LOCATION}")
print(f"   Isolierter Azure CLI Kontext: {NOTEBOOK_AZURE_CONFIG}")

def az_cmd(args):
    """Run Azure CLI via cmd.exe (works with az.cmd shims on Windows)."""
    return run_args(["cmd", "/c"] + args)

# Login with Service Principal
rc = az_cmd([
    "az", "login", "--service-principal",
    "-u", CLIENT_ID,
    f"--password={CLIENT_SECRET}",
    "--tenant", TENANT_ID,
    "--only-show-errors", "--output", "none",
])
if rc != 0:
    raise SystemExit("❌ SP Login fehlgeschlagen")

# Set subscription to VM subscription
rc = az_cmd(["az", "account", "set", "-s", VM_SUBSCRIPTION_ID, "--only-show-errors", "--output", "none"])
if rc != 0:
    raise SystemExit("❌ az account set fehlgeschlagen")

current_account = run_output("az account show --query user.name -o tsv --only-show-errors")
print(f"✅ SP Login erfolgreich: {current_account}")

✅ Konfiguration geladen
   Tenant ID: f71980b2-590a-4de9-90d5-6fbc867da951
   VM Subscription: 09808f31-065f-4231-914d-776c2d6bbe34
   ODAA Subscription: 4aecf0e8-2fe2-4187-bc93-0356bd2676f5
   Location: francecentral
   Isolierter Azure CLI Kontext: c:\Users\chpinoto\workspace\msftmh\03-Azure\01-03-Infrastructure\10_Oracle_on_Azure\resources\infra\terraform\.azure-notebook
✅ SP Login erfolgreich: 8a9f736e-4eb2-4484-ae90-2493f57102b3
✅ Secret aus Notebook-Variablen entfernt


## Manage Users (Password Rotation + MFA Reset)

Kombiniertes Script `manage-users.ps1` für Workshop-User-Verwaltung:
- **rotate-passwords**: Neue Passwörter generieren via `az ad user update` (kein Terraform nötig)
- **reset-mfa**: Alle MFA-Methoden entfernen (Authenticator, Phone, etc.)
- **reset-all**: Beides in einem Durchlauf

Die Credentials werden direkt in `user_credentials.json` aktualisiert.

In [5]:
import subprocess, os, json

# Action: "rotate-passwords" | "reset-mfa" | "reset-all"
ACTION = "reset-all"

# Optional: Event name for metadata in user_credentials.json
EVENT_NAME = "workshop-march-2026"

script_path = os.path.join(TERRAFORM_DIR, "scripts", "manage-users.ps1")
cred_file = os.path.join(TERRAFORM_DIR, "user_credentials.json")

# Fallback to identity folder
if not os.path.exists(cred_file):
    cred_file = os.path.join(TERRAFORM_DIR, "identity", "user_credentials.json")

cmd = f'pwsh -File "{script_path}" -Action {ACTION} -CredentialsFile "{cred_file}"'
if EVENT_NAME:
    cmd += f' -EventName "{EVENT_NAME}"'

print(f"Running: {cmd}\n")
rc = subprocess.call(cmd, shell=True)

if rc == 0 and ACTION in ("rotate-passwords", "reset-all"):
    # Display updated credentials
    if os.path.exists(cred_file):
        with open(cred_file, "r", encoding="utf-8") as f:
            creds = json.load(f)
        print("\n" + "=" * 70)
        print("Aktuelle User Credentials")
        print("=" * 70)
        for user_key, user_data in creds.get("users", {}).items():
            upn = user_data.get("user_principal_name", "?")
            pwd = user_data.get("password", "?")
            print(f"  {upn}  →  {pwd}")
        print("=" * 70)
elif rc != 0:
    print(f"\n❌ Fehlgeschlagen (rc={rc})")

Running: pwsh -File "c:\Users\chpinoto\workspace\msftmh\03-Azure\01-03-Infrastructure\10_Oracle_on_Azure\resources\infra\terraform\scripts\manage-users.ps1" -Action reset-all -CredentialsFile "c:\Users\chpinoto\workspace\msftmh\03-Azure\01-03-Infrastructure\10_Oracle_on_Azure\resources\infra\terraform\user_credentials.json" -EventName "workshop-march-2026"


❌ Fehlgeschlagen (rc=1)


## 3. Get User00 VM Information

Hole die VM-Informationen für User00 aus den Terraform Outputs oder Azure.

In [3]:
# ============================================================================
# Get User00 VM Information
# ============================================================================

USER_INDEX = "00"
VM_NAME = f"vm-user{USER_INDEX}"
RG_NAME = f"rg-vm-user{USER_INDEX}"

# Get VM details from Azure
print(f"🔍 Suche VM: {VM_NAME} in RG: {RG_NAME}")

vm_info = az_json(f'az vm show -g {RG_NAME} -n {VM_NAME} -o json --only-show-errors')
if not vm_info:
    raise SystemExit(f"❌ VM {VM_NAME} nicht gefunden. Stelle sicher dass 'terraform apply' ausgeführt wurde.")

# Get public IP
public_ip = run_output(f'az vm show -g {RG_NAME} -n {VM_NAME} -d --query publicIps -o tsv --only-show-errors')
private_ip = run_output(f'az vm show -g {RG_NAME} -n {VM_NAME} -d --query privateIps -o tsv --only-show-errors')

print(f"✅ VM gefunden:")
print(f"   Name: {VM_NAME}")
print(f"   Resource Group: {RG_NAME}")
print(f"   Public IP: {public_ip}")
print(f"   Private IP: {private_ip}")
print(f"   Location: {vm_info.get('location', 'unknown')}")

# Store for later use
VM_PUBLIC_IP = public_ip
VM_PRIVATE_IP = private_ip

🔍 Suche VM: vm-user00 in RG: rg-vm-user00
✅ VM gefunden:
   Name: vm-user00
   Resource Group: rg-vm-user00
   Public IP: 4.233.91.238
   Private IP: 10.0.0.4
   Location: francecentral


## 4. SSH via Azure CLI (Entra ID)

Verbindung zur VM via `az ssh vm` mit dem Service Principal.
Der SP hat `Virtual Machine Administrator Login` Rechte.

In [4]:
# ============================================================================
# SSH Connection via Azure CLI (Entra ID Authentication) - non-interactive
# ============================================================================
# Ziel: SSH schnell testen, ohne dass irgendwas "hängen" kann (Host-Key Prompt etc.).
# ============================================================================

print("=" * 70)
print("SSH Verbindung zur User00 VM")
print("=" * 70)

if "run" not in globals():
    raise SystemExit("❌ Helper-Funktion run() fehlt. Bitte zuerst Zelle 3 ausführen.")
if "VM_PUBLIC_IP" not in globals() or not VM_PUBLIC_IP:
    raise SystemExit("❌ VM_PUBLIC_IP fehlt. Bitte zuerst Zelle 7 ausführen.")

# Check if ssh extension is installed (timeout)
print("\n🔍 Prüfe ob Azure CLI SSH Extension installiert ist...")
try:
    rc = run("az extension show --name ssh --only-show-errors -o none", shell=True, timeout=30)
except subprocess.TimeoutExpired:
    raise SystemExit("❌ Timeout bei 'az extension show' (Azure CLI hängt?)")

if rc != 0:
    print("📦 Installiere SSH Extension...")
    run("az extension add --name ssh --only-show-errors", shell=True, timeout=120)

# Non-interactive SSH options to avoid prompts/hangs
SSH_OPTS = "-o BatchMode=yes -o ConnectTimeout=20 -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null"

print(f"\n🔗 Teste SSH Verbindung zu {VM_PUBLIC_IP} (non-interactive, timeout)...")
print("-" * 70)

ssh_test_cmd = f'az ssh vm --ip {VM_PUBLIC_IP} -- {SSH_OPTS} "echo OK && hostname && whoami"'

try:
    rc = run(ssh_test_cmd, shell=True, timeout=90)
except subprocess.TimeoutExpired:
    raise SystemExit("❌ Timeout: SSH Test >90s (Port 22 / AADSSHLoginForLinux / NSG prüfen)")

print("-" * 70)
if rc == 0:
    print("✅ SSH Verbindung erfolgreich!")
else:
    print("⚠️  SSH Verbindung fehlgeschlagen. Mögliche Ursachen:")
    print("   - NSG blockiert SSH (Port 22)")
    print("   - AADSSHLoginForLinux Extension nicht installiert/fehlt")
    print("   - SP hat keine 'Virtual Machine Administrator Login' Rolle")
    print("   - Azure CLI ssh-extension Problem")

SSH Verbindung zur User00 VM

🔍 Prüfe ob Azure CLI SSH Extension installiert ist...

🔗 Teste SSH Verbindung zu 4.233.91.238 (non-interactive, timeout)...
----------------------------------------------------------------------
----------------------------------------------------------------------
✅ SSH Verbindung erfolgreich!


OpenSSH_for_Windows_9.5p2, LibreSSL 3.8.2
8a9f736e-4eb2-4484-ae90-2493f57102b3@4.233.91.238: Permission denied (publickey).



## 5. Check Oracle Tools auf der VM

Prüfe ob die Oracle Tools (SQLcl, Instant Client, etc.) auf der VM installiert sind.

In [5]:
# ============================================================================
# Check Oracle Tools on VM
# ============================================================================

print("=" * 70)
print("Oracle Tools Check auf der VM")
print("=" * 70)

check_commands = """
echo "=== Oracle Instant Client ==="
ls -la /opt/oracle/instantclient* 2>/dev/null || echo "Not found"
echo ""
echo "=== SQLcl ==="
/opt/oracle/sqlcl/bin/sql -V 2>/dev/null || echo "Not found"
echo ""
echo "=== Java ==="
java -version 2>&1 | head -1
echo ""
echo "=== Azure CLI ==="
az --version 2>/dev/null | head -1
echo ""
echo "=== OCI CLI ==="
oci --version 2>/dev/null || echo "Not found"
echo ""
echo "=== rwloadsim/connping ==="
which connping 2>/dev/null || echo "Not found"
echo ""
echo "=== Network Tools ==="
which dig traceroute nc tcpdump 2>/dev/null | head -5
"""

rc = run(f'az ssh vm --ip {VM_PUBLIC_IP} -- "{check_commands}"', shell=True)

if rc == 0:
    print("-" * 70)
    print("✅ Oracle Tools Check abgeschlossen")
else:
    print("⚠️  Check fehlgeschlagen")

Oracle Tools Check auf der VM
----------------------------------------------------------------------
✅ Oracle Tools Check abgeschlossen


OpenSSH_for_Windows_9.5p2, LibreSSL 3.8.2
Pseudo-terminal will not be allocated because stdin is not a terminal.
@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
@    WARNING: REMOTE HOST IDENTIFICATION HAS CHANGED!     @
@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
IT IS POSSIBLE THAT SOMEONE IS DOING SOMETHING NASTY!
Someone could be eavesdropping on you right now (man-in-the-middle attack)!
It is also possible that a host key has just been changed.
The fingerprint for the ED25519 key sent by the remote host is
SHA256:oF299DCij6KM91me5EYxE50CqrjlnC90gkOgORVngiE.
Please contact your system administrator.
Add correct host key in C:\\Users\\chpinoto/.ssh/known_hosts to get rid of this message.
Offending ECDSA key in C:\\Users\\chpinoto/.ssh/known_hosts:125
Host key for 20.111.56.212 has changed and you have requested strict checking.
Host key verification failed.



## 6. Test ODAA VNet Connectivity

Teste die Netzwerkverbindung zum ODAA VNet (192.168.x.x) über das VNet Peering.

In [ ]:
# ============================================================================
# Test ODAA VNet Connectivity
# ============================================================================
# Das VM VNet (10.0.0.0/16) ist mit dem ODAA VNet (192.168.0.0/16) gepeert.
# Die ADB hat typischerweise eine IP im Bereich 192.168.0.x
# ============================================================================

print("=" * 70)
print("ODAA VNet Connectivity Test")
print("=" * 70)

# Get ODAA VNet information
ODAA_RG = f"rg-odaa-user{USER_INDEX}"
ODAA_VNET = f"vnet-odaa-user{USER_INDEX}"

print(f"\n🔍 Hole ODAA VNet Informationen...")
print(f"   Resource Group: {ODAA_RG}")
print(f"   VNet: {ODAA_VNET}")

# Switch to ODAA subscription temporarily
print(f"\n🔄 Wechsle zu ODAA Subscription...")
run(f'az account set -s {ODAA_SUBSCRIPTION_ID} --only-show-errors', shell=True)

# Get ODAA subnet info
odaa_vnet_info = az_json(f'az network vnet show -g {ODAA_RG} -n {ODAA_VNET} -o json --only-show-errors')
if odaa_vnet_info:
    address_space = odaa_vnet_info.get('addressSpace', {}).get('addressPrefixes', [])
    subnets = odaa_vnet_info.get('subnets', [])
    print(f"✅ ODAA VNet gefunden:")
    print(f"   Address Space: {address_space}")
    for subnet in subnets:
        print(f"   Subnet: {subnet.get('name')} - {subnet.get('addressPrefix')}")
else:
    print(f"⚠️  ODAA VNet {ODAA_VNET} nicht gefunden")

# Switch back to VM subscription
run(f'az account set -s {VM_SUBSCRIPTION_ID} --only-show-errors', shell=True)

# Test connectivity from VM to ODAA subnet
print(f"\n🔗 Teste Netzwerk-Konnektivität von VM zu ODAA VNet...")
print("-" * 70)

connectivity_test = """
echo "=== Network Interfaces ==="
ip addr show | grep -E "inet |^[0-9]"
echo ""
echo "=== Route Table ==="
ip route
echo ""
echo "=== DNS Resolution Test ==="
# Try to resolve Oracle DNS zone
nslookup adb.eu-paris-1.oraclecloud.com 2>/dev/null || echo "DNS lookup failed"
echo ""
echo "=== Ping Test to ODAA Subnet (192.168.0.0/24) ==="
# Ping gateway of ODAA subnet
ping -c 3 -W 2 192.168.0.1 2>/dev/null || echo "Ping to 192.168.0.1 failed (may be blocked by NSG)"
"""

rc = run(f'az ssh vm --ip {VM_PUBLIC_IP} -- "{connectivity_test}"', shell=True)
print("-" * 70)
print("✅ Connectivity Test abgeschlossen")

## 7. ADB Connection (SQLcl)

Verbindung zur Oracle Autonomous Database via SQLcl.
Der TNS Connection String muss aus dem OCI Portal geholt werden.

### Vorgehen (kleine Schritte) – erst SSH, dann ADB/SQLcl

.
**Wichtig:** Wenn irgendwas „hängt“, liegt es fast immer schon an der SSH-Strecke (`az ssh vm …`) oder an interaktiven Prompts. Deshalb: **immer zuerst SSH zuverlässig hinbekommen**, erst danach SQLcl/ADB testen.

#### Schritt 1: Notebook-Setup (muss nach Kernel-Restart)
- Zelle 3 ausführen (Helper)
- Zelle 5 ausführen (Service Principal Login)
- Zelle 7 ausführen (VM IP holen)
  - Erwartung: `VM_PUBLIC_IP` wird angezeigt

.
#### Schritt 2: SSH Sanity Check (muss schnell „OK“ liefern)
- Zelle 10 ausführen (SSH non-interactive)
  - Erwartung: Ausgabe enthält `OK`, `hostname`, `whoami`
  - Wenn Zelle 10 nicht sauber durchläuft: **ADB noch nicht testen** → erst SSH/NSG/Extension fixen.

.
Typische Ursachen wenn SSH hängt/fehlschlägt:
- NSG blockiert Port 22 zur VM
- VM-Extension `AADSSHLoginForLinux` fehlt oder ist defekt
- Rolle fehlt: `Virtual Machine Administrator Login` (oder User Login)
- Azure CLI `ssh` extension Problem / lokaler SSH Client hängt

.
#### Schritt 3: Oracle Tools auf der VM prüfen
- Zelle 11 ausführen (Oracle Tools Check)
  - Erwartung: `instantclient` und `sql -V` sind vorhanden

.
#### Schritt 4: Netzwerk/DNS zur ADB (von der VM aus)
- (Optional) In Zelle 13 (Connectivity) DNS-Teil prüfen, oder per SSH manuell ausführen:
  - `nslookup <adb-host-aus-dem-tns>`
  - Test Port (typisch 1521): `nc -vz <adb-host> 1521` (falls `nc` vorhanden)
- Wenn DNS/Port nicht klappt: Private DNS Zone / VNet-Peering / NSG / Routes fixen.

.
#### Schritt 5: SQLcl Test gegen ADB
- Danach erst Zelle 15 ausführen (ADB Connection)
  - Nutzt den **kompletten** TNS Descriptor (die `(description=...)` Klammer)
  - Wenn möglich Passwort nicht ins Notebook schreiben: lieber als Environment Variable `ADB_PASSWORD` setzen.

---
Wenn du magst: poste einfach die Ausgabe von Schritt 2 (Zelle 10) – damit können wir die Fehlerursache sehr schnell eingrenzen.

In [ ]:
# ============================================================================
# ADB Connection via SQLcl (small steps + timeouts)
# ============================================================================
# Schritt 1: Prüft SSH (non-interactive) + Oracle Tools auf der VM.
# Schritt 2: Führt dann erst den SQLcl Test aus (mit Timeout).
# ============================================================================

import os
import subprocess
import base64
import textwrap

if "VM_PUBLIC_IP" not in globals() or not VM_PUBLIC_IP:
    raise SystemExit("❌ VM_PUBLIC_IP fehlt. Bitte zuerst Zelle 7 (VM Info) ausführen.")

# --- Configure here ---
ADB_TNS_CONNECTION_STRING = "(description= (retry_count=20)(retry_delay=3)(address=(protocol=tcps)(port=1521)(host=t5tdbqxk.adb.eu-paris-1.oraclecloud.com))(connect_data=(service_name=gc2401553d1c7ab_test_high.adb.oraclecloud.com))(security=(ssl_server_dn_match=no)))"
ADB_USER = "ADMIN"
ADB_PASSWORD = ""  # ⚠️ leer lassen und per Env ADB_PASSWORD setzen ODER hier eintragen (nicht committen!)

if not ADB_PASSWORD:
    # prefer env var to avoid storing secrets in the notebook file
    ADB_PASSWORD = os.environ.get("ADB_PASSWORD", "").strip()
if not ADB_PASSWORD:
    raise SystemExit("❌ Bitte ADB_PASSWORD setzen (Empfehlung: Environment Variable ADB_PASSWORD).")

def bash_sq(s: str) -> str:
    return "'" + s.replace("'", "'\"'\"'") + "'"

print("=" * 70)
print("ADB Connection Test (SQLcl on VM)")
print("=" * 70)
print(f"VM IP: {VM_PUBLIC_IP}")
print("(Passwort wird nicht angezeigt)")

# Non-interactive SSH options to avoid host key prompts
SSH_OPTS = "-o BatchMode=yes -o ConnectTimeout=20 -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null"

# 0) Quick SSH sanity check (fast fail)
payload0 = base64.b64encode(b"echo OK; hostname; whoami\n").decode("ascii")
cmd0 = f"printf %s {bash_sq(payload0)} | base64 -d | bash"
cmdline0 = f'az ssh vm --ip {VM_PUBLIC_IP} -- {SSH_OPTS} "{cmd0}"'

try:
    p0 = subprocess.run(["cmd", "/c", cmdline0], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=90)
except subprocess.TimeoutExpired:
    raise SystemExit("❌ Timeout: SSH sanity check >90s (erst SSH/NSG/AADSSHLoginForLinux fixen)")

if p0.stdout:
    print(p0.stdout)
if p0.stderr:
    print(p0.stderr)
if p0.returncode != 0:
    raise SystemExit(f"❌ SSH sanity check failed (rc={p0.returncode}). Erst SSH beheben, dann SQLcl testen.")

# 1) SQLcl test (runs on VM)
remote_script = textwrap.dedent(f"""\
set -euo pipefail

ORACLE_HOME="$(ls -d /opt/oracle/instantclient_* 2>/dev/null | sort | tail -1 || true)"
if [ -z "$ORACLE_HOME" ]; then
  echo "❌ Oracle Instant Client nicht gefunden unter /opt/oracle/instantclient_*" >&2
  exit 2
fi

export ORACLE_HOME
export LD_LIBRARY_PATH="$ORACLE_HOME:${{LD_LIBRARY_PATH:-}}"
export PATH="/opt/oracle/sqlcl/bin:$PATH"

if ! command -v sql >/dev/null 2>&1; then
  echo "❌ SQLcl (sql) nicht gefunden unter /opt/oracle/sqlcl/bin" >&2
  exit 3
fi

CONN={bash_sq(ADB_TNS_CONNECTION_STRING)}
USR={bash_sq(ADB_USER)}
PWD={bash_sq(ADB_PASSWORD)}

echo "✅ Oracle Client: $ORACLE_HOME"
echo "✅ SQLcl: $(sql -V 2>/dev/null || true)"
echo "-- Running test query (timeout 90s)"

SQL_CMD="sql -S \"${{USR}}/\\\"${{PWD}}\\\"@\\\"${{CONN}}\\\"\""
if command -v timeout >/dev/null 2>&1; then
  echo "SELECT 1 AS status FROM dual;" | timeout 90 bash -lc "$SQL_CMD"
else
  echo "SELECT 1 AS status FROM dual;" | bash -lc "$SQL_CMD"
fi
""")

payload = base64.b64encode(remote_script.encode("utf-8")).decode("ascii")
remote_cmd = f"printf %s {bash_sq(payload)} | base64 -d | bash"
cmdline = f'az ssh vm --ip {VM_PUBLIC_IP} -- {SSH_OPTS} "{remote_cmd}"'

try:
    p = subprocess.run(["cmd", "/c", cmdline], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=180)
except subprocess.TimeoutExpired:
    raise SystemExit("❌ Timeout: az ssh vm / SQLcl Test >180s")

if p.stdout:
    print(p.stdout)
if p.stderr:
    print(p.stderr)

if p.returncode == 0:
    print("✅ ADB Verbindung erfolgreich!")
else:
    print(f"⚠️  ADB Verbindung fehlgeschlagen (rc={p.returncode}).")

ADB Connection Test (SQLcl on VM)
VM IP: 20.111.56.212
(Passwort wird nicht angezeigt)


## 8. Interactive SSH Session

Öffne eine interaktive SSH-Session zur VM.
Führe diesen Befehl im Terminal aus (nicht im Notebook):

In [1]:
# ============================================================================
# Interactive SSH Session Commands
# ============================================================================
# Kopiere diese Befehle und führe sie im Terminal aus.
# Das Notebook kann keine interaktiven Sessions starten.
# ============================================================================

print("=" * 70)
print("SSH Befehle für Terminal")
print("=" * 70)

print(f"""
1️⃣  SSH Verbindung zur VM (via Azure CLI + Entra ID):
   
   az ssh vm --ip {VM_PUBLIC_IP}

2️⃣  Sobald auf der VM, SQLcl starten:
   
   # Oracle Environment setzen
   export ORACLE_HOME=/opt/oracle/instantclient_23_5
   export LD_LIBRARY_PATH=$ORACLE_HOME:$LD_LIBRARY_PATH
   export PATH=/opt/oracle/sqlcl/bin:$PATH
   
   # SQLcl starten (ersetze <TNS> mit deinem Connection String)
   sql ADMIN@"<HOST>:1522/<TNS_NAME>"

3️⃣  Alternativ: Connping für Connectivity-Test:
   
   connping <ADB_HOST> 1522

4️⃣  DNS Lookup für ADB:
   
   nslookup adb.eu-paris-1.oraclecloud.com
   dig +short @168.63.129.16 adb.eu-paris-1.oraclecloud.com
""")

print("=" * 70)
print(f"✅ VM Public IP: {VM_PUBLIC_IP}")
print(f"✅ VM Private IP: {VM_PRIVATE_IP}")
print("=" * 70)

SSH Befehle für Terminal


NameError: name 'VM_PUBLIC_IP' is not defined